# Notebook 2: Data Cleaning & Feature Engineering

# Hotel Revenue Management & Dynamic Pricing Intelligence

---

## Project Workflow

- Notebook 1 — Business Understanding & Data Audit
- ✅ Notebook 2 — Data Cleaning & Feature Engineering *(Current Notebook)*
- Notebook 3 — Exploratory Data Analysis
- Notebook 4 — SQL Business Analysis
- Notebook 5 — Revenue Management Simulation

---

## Objective

The objective of this notebook is to prepare the raw reservation dataset for downstream analytics by handling missing values, validating data quality, converting data types, and engineering new business features required for occupancy modelling, pricing simulation, and revenue analysis.

In [1]:
# ==========================================================
# 1. Import Libraries
# ==========================================================

import pandas as pd
import numpy as np

from datetime import timedelta

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==========================================================
# 2. Load Raw Dataset
# ==========================================================

df = pd.read_csv("../data/raw/hotel_bookings.csv")

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

df.head()

Rows    : 119,390
Columns : 32


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [3]:
# ==========================================================
# 3. Initial Validation
# ==========================================================

print("Missing Values\n")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDuplicate Rows")
print(df.duplicated().sum())

Missing Values

children         4
country        488
agent        16340
company     112593
dtype: int64

Duplicate Rows
31994


## 1. Handle Missing Values

The first step is to evaluate whether missing values represent genuine data quality issues or expected business scenarios.

Rather than applying a single imputation strategy, each feature will be assessed individually to preserve the business meaning of the data.

In [4]:
# ==========================================================
# Missing Values
# ==========================================================

missing_summary = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum()/len(df)*100).round(2)
})

missing_summary = (
    missing_summary[missing_summary["Missing Values"] > 0]
    .sort_values("Missing Values", ascending=False)
)

missing_summary

,Missing Values,Missing Percentage
company,112593,94.31
agent,16340,13.69
country,488,0.41
children,4,0.00


In [5]:
# ==========================================================
# Handle Missing Values
# ==========================================================

# Children: Missing implies no children accompanied the booking
df["children"] = df["children"].fillna(0)

# Country: Preserve reservations with unknown country
df["country"] = df["country"].fillna("Unknown")

### Observations

- Missing values in `children` were replaced with `0`.
- Missing values in `country` were labelled as `"Unknown"`.
- Missing values in `agent` and `company` were intentionally retained.

### Business Interpretation

Missing values in `agent` and `company` provide useful business information rather than indicating poor data quality.

A missing `agent` typically represents a direct booking, while a missing `company` suggests the reservation was not made through a corporate account.

Retaining these missing values preserves information that will later be used for customer segmentation and booking channel analysis.

## 2. Validate Duplicate Records

Duplicate reservations identified during the initial data audit are re-evaluated before any cleaning operation.

Since the dataset does not include a unique reservation identifier, identical rows may represent independent bookings made by different guests rather than data entry errors.

Therefore, duplicate records are retained to preserve genuine booking demand.

In [7]:
# ==========================================================
# Duplicate Validation
# ==========================================================

duplicate_count = df.duplicated().sum()

print(f"Duplicate Records Identified : {duplicate_count:,}")


Duplicate Records Identified : 31,994


### Observations

- A total of **31,994** identical records were identified.
- No unique booking identifier is available to distinguish duplicate entries from legitimate reservations.

### Business Interpretation

Removing identical reservations may incorrectly eliminate valid bookings because multiple guests can legitimately share the same booking characteristics.

To preserve booking demand and avoid introducing analytical bias, all reservations are retained for subsequent analysis.

## 3. Date Feature Engineering

Reservation dates are currently stored across multiple columns. For time-based analysis, occupancy reconstruction, and integration with external datasets such as weather and public holidays, these attributes are consolidated into complete date fields.

This section creates the core temporal features that will be used throughout the remainder of the project.

In [8]:
# ==========================================================
# Create Arrival Date
# ==========================================================

df["arrival_date"] = pd.to_datetime(
    df["arrival_date_year"].astype(str) + "-" +
    df["arrival_date_month"] + "-" +
    df["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d"
)

df["arrival_date"].head()

0   2015-07-01
1   2015-07-01
2   2015-07-01
3   2015-07-01
4   2015-07-01
Name: arrival_date, dtype: datetime64[ns]

In [9]:
# ==========================================================
# Create Stay Duration
# ==========================================================

df["stay_duration"] = (
    df["stays_in_week_nights"] +
    df["stays_in_weekend_nights"]
)

df["stay_duration"].describe()

count    119390.000000
mean          3.427900
std           2.557439
min           0.000000
25%           2.000000
50%           3.000000
75%           4.000000
max          69.000000
Name: stay_duration, dtype: float64

In [10]:
# ==========================================================
# Create Departure Date
# ==========================================================

df["departure_date"] = (
    df["arrival_date"] +
    pd.to_timedelta(df["stay_duration"], unit="D")
)

df[["arrival_date", "departure_date"]].head()

,arrival_date,departure_date
0,2015-07-01,2015-07-01
1,2015-07-01,2015-07-01
2,2015-07-01,2015-07-02
3,2015-07-01,2015-07-02
4,2015-07-01,2015-07-03


In [11]:
df[
    [
        "arrival_date",
        "stay_duration",
        "departure_date"
    ]
].sample(5, random_state=42)

,arrival_date,stay_duration,departure_date
30946,2016-12-02,7,2016-12-09
40207,2015-07-16,3,2015-07-19
103708,2016-12-27,3,2016-12-30
85144,2016-03-09,1,2016-03-10
109991,2017-04-16,4,2017-04-20


### Business Interpretation

These derived date features form the foundation of the revenue management analysis.

- **Arrival Date** enables time-series analysis and integration with external demand drivers.
- **Stay Duration** supports occupancy calculations and revenue estimation.
- **Departure Date** is essential for reconstructing daily room occupancy, which is not directly available in the source dataset.

## 4. Customer & Booking Features

This section derives business-oriented features that simplify customer segmentation and booking analysis.

These engineered attributes improve readability and eliminate the need to repeatedly calculate common business metrics in later notebooks.

In [12]:
# ==========================================================
# Create Total Guests
# ==========================================================

df["total_guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
).astype(int)

df["total_guests"].describe()

count    119390.000000
mean          1.968239
std           0.722394
min           0.000000
25%           2.000000
50%           2.000000
75%           2.000000
max          55.000000
Name: total_guests, dtype: float64

### Business Interpretation

Total guest count provides a more intuitive measure of booking size than analysing adults, children, and babies separately. This feature will support customer segmentation, occupancy estimation, and demand analysis.

In [13]:
# ==========================================================
# Create Booking Source
# ==========================================================

df["booking_source"] = np.select(
    [
        df["company"].notna(),
        df["agent"].notna()
    ],
    [
        "Corporate",
        "Travel Agent"
    ],
    default="Direct"
)

df["booking_source"].value_counts()

booking_source
Travel Agent    102833
Direct            9760
Corporate         6797
Name: count, dtype: int64

In [14]:
# ==========================================================
# Create Lead Time Buckets
# ==========================================================

bins = [-1, 7, 30, 90, np.inf]

labels = [
    "0-7 Days",
    "8-30 Days",
    "31-90 Days",
    "90+ Days"
]

df["lead_time_bucket"] = pd.cut(
    df["lead_time"],
    bins=bins,
    labels=labels
)

df["lead_time_bucket"].value_counts().sort_index()

lead_time_bucket
0-7 Days      19746
8-30 Days     18960
31-90 Days    29553
90+ Days      51131
Name: count, dtype: int64

### Business Interpretation

Grouping reservations into lead-time windows simplifies demand analysis and forms the basis for the dynamic pricing strategy developed later in the project.

In [15]:
# ==========================================================
# Create Arrival Season
# ==========================================================

season_map = {
    "December": "Winter",
    "January": "Winter",
    "February": "Winter",

    "March": "Spring",
    "April": "Spring",
    "May": "Spring",

    "June": "Summer",
    "July": "Summer",
    "August": "Summer",

    "September": "Autumn",
    "October": "Autumn",
    "November": "Autumn"
}

df["arrival_season"] = (
    df["arrival_date_month"]
      .map(season_map)
)

df["arrival_season"].value_counts()

arrival_season
Summer    37477
Spring    32674
Autumn    28462
Winter    20777
Name: count, dtype: int64

## 5. Arrival Behaviour Features

Customer arrival patterns often differ between weekdays and weekends. This section derives features that help identify booking behaviour across different periods of the week and seasons.

These features will later support occupancy analysis, pricing strategies, and demand forecasting.

In [16]:
# ==========================================================
# Create Weekend Arrival Flag
# ==========================================================

df["weekend_arrival"] = np.where(
    df["arrival_date"].dt.dayofweek >= 5,
    "Weekend",
    "Weekday"
)

df["weekend_arrival"].value_counts()

weekend_arrival
Weekday    87194
Weekend    32196
Name: count, dtype: int64

### Business Interpretation

Weekend arrivals often exhibit different booking behaviour, occupancy patterns, and pricing dynamics compared to weekday arrivals.

This feature will support demand analysis and dynamic pricing recommendations in later stages of the project.

In [18]:
# ==========================================================
# Create Arrival Month Number
# ==========================================================

df["arrival_month_number"] = (
    df["arrival_date"].dt.month
)

df[
    [
        "arrival_date_month",
        "arrival_month_number"
    ]
].head()

,arrival_date_month,arrival_month_number
0,July,7
1,July,7
2,July,7
3,July,7
4,July,7


In [19]:
# ==========================================================
# Create Arrival Day Name
# ==========================================================

df["arrival_day_name"] = (
    df["arrival_date"].dt.day_name()
)

df["arrival_day_name"].value_counts()

arrival_day_name
Friday       19631
Thursday     19254
Monday       18171
Saturday     18055
Wednesday    16139
Sunday       14141
Tuesday      13999
Name: count, dtype: int64

In [20]:
df[
    [
        "arrival_date",
        "arrival_day_name",
        "weekend_arrival",
        "arrival_month_number",
        "arrival_season"
    ]
].sample(10, random_state=42)

,arrival_date,arrival_day_name,weekend_arrival,arrival_month_number,arrival_season
30946,2016-12-02,Friday,Weekday,12,Winter
40207,2015-07-16,Thursday,Weekday,7,Summer
103708,2016-12-27,Tuesday,Weekday,12,Winter
85144,2016-03-09,Wednesday,Weekday,3,Spring
109991,2017-04-16,Sunday,Weekend,4,Spring
110622,2017-04-27,Thursday,Weekday,4,Spring
47790,2016-03-04,Friday,Weekday,3,Spring
44992,2015-10-16,Friday,Weekday,10,Autumn
30528,2016-11-19,Saturday,Weekend,11,Autumn
16886,2015-09-14,Monday,Weekday,9,Autumn


## 6. Business Metrics

To simplify downstream analysis, several operational metrics are derived from the reservation data.

These features provide a standardized basis for occupancy analysis, pricing evaluation, and revenue calculations in later notebooks.

In [24]:
# ==========================================================
# Create Room Nights
# ==========================================================

df["room_nights"] = df["stay_duration"]

df["room_nights"].describe()

count    119390.000000
mean          3.427900
std           2.557439
min           0.000000
25%           2.000000
50%           3.000000
75%           4.000000
max          69.000000
Name: room_nights, dtype: float64

### Business Interpretation

Room Nights represent the total number of nights a room is occupied for a reservation.

This metric forms the basis for occupancy calculations and revenue estimation.

In [21]:
# ==========================================================
# Create Guest Nights
# ==========================================================

df["guest_nights"] = (
    df["total_guests"] *
    df["stay_duration"]
)

df["guest_nights"].describe()

count    119390.000000
mean          6.960240
std           6.132987
min           0.000000
25%           3.000000
50%           6.000000
75%           8.000000
max         182.000000
Name: guest_nights, dtype: float64

### Business Interpretation

Guest Nights measure the total number of guest-days generated by a reservation.

This metric is commonly used to evaluate hotel demand and customer volume.

In [22]:
# ==========================================================
# Weekend Stay Indicator
# ==========================================================

df["weekend_stay"] = np.where(
    df["stays_in_weekend_nights"] > 0,
    "Yes",
    "No"
)

df["weekend_stay"].value_counts()

weekend_stay
Yes    67392
No     51998
Name: count, dtype: int64

In [25]:
# Validation
df[
    [
        "stay_duration",
        "room_nights",
        "guest_nights",
        "weekend_stay"
    ]
].sample(10, random_state=42)

,stay_duration,room_nights,guest_nights,weekend_stay
30946,7,7,14,Yes
40207,3,3,6,No
103708,3,3,9,No
85144,1,1,1,No
109991,4,4,8,Yes
110622,3,3,6,No
47790,1,1,1,No
44992,2,2,4,No
30528,7,7,14,Yes
16886,2,2,4,Yes


In [26]:
# ==========================================================
# Create Booking Status
# ==========================================================

df["booking_status"] = df["reservation_status"].replace({
    "Check-Out": "Completed",
    "Canceled": "Cancelled",
    "No-Show": "No Show"
})

df["booking_status"].value_counts()

booking_status
Completed    75166
Cancelled    43017
No Show       1207
Name: count, dtype: int64

In [30]:
# ==========================================================
# Create Family Type
# ==========================================================

df["family_type"] = np.select(
    [
        df["total_guests"] == 1,
        df["total_guests"] == 2,
        df["total_guests"] > 2
    ],
    [
        "Solo",
        "Couple",
        "Group"
    ],
    default="Unknown"
)

df["family_type"].value_counts()

family_type
Couple     82051
Solo       22581
Group      14578
Unknown      180
Name: count, dtype: int64

In [28]:
# ==========================================================
# Create Arrival Quarter
# ==========================================================

df["arrival_quarter"] = (
    "Q" +
    df["arrival_date"].dt.quarter.astype(str)
)

df["arrival_quarter"].value_counts().sort_index()

arrival_quarter
Q1    23791
Q2    33819
Q3    37046
Q4    24734
Name: count, dtype: int64

In [29]:
# ==========================================================
# Create Arrival Year-Month
# ==========================================================

df["arrival_year_month"] = (
    df["arrival_date"].dt.to_period("M").astype(str)
)

df["arrival_year_month"].head()

0    2015-07
1    2015-07
2    2015-07
3    2015-07
4    2015-07
Name: arrival_year_month, dtype: object

### Business Interpretation

The engineered business features simplify customer segmentation and time-based reporting.

- **Booking Status** standardizes reservation outcomes for easier analysis.
- **Family Type** groups reservations by party size, enabling customer segmentation.
- **Arrival Quarter** supports quarterly performance reporting and executive KPIs.
- **Arrival Year-Month** provides a standardized time dimension for monthly trend analysis in Python, SQL, and Tableau.

These features reduce repetitive transformations in later notebooks and create a consistent analytical dataset for downstream business intelligence.

## 7. Data Type Conversion

Several attributes require data type conversion before feature engineering.

Converting columns to their appropriate data types improves analytical consistency, enables time-based calculations, and reduces memory usage.

In [34]:
# ==========================================================
# Convert Date Columns
# ==========================================================

df["reservation_status_date"] = pd.to_datetime(
    df["reservation_status_date"]
)

print(df["reservation_status_date"].dtype)

datetime64[ns]


In [35]:
# ==========================================================
# Convert Object Columns to Category
# ==========================================================

category_columns = [
    "hotel",
    "meal",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "assigned_room_type",
    "deposit_type",
    "customer_type",
    "reservation_status",
    "country"
]

for col in category_columns:
    df[col] = df[col].astype("category")

df[category_columns].dtypes

hotel                   category
meal                    category
market_segment          category
distribution_channel    category
reserved_room_type      category
assigned_room_type      category
deposit_type            category
customer_type           category
reservation_status      category
country                 category
dtype: object

## 8. Final Dataset Validation

Before exporting the cleaned dataset, a final validation is performed to ensure that all cleaning and feature engineering steps have been applied successfully.

The validation confirms dataset completeness, verifies data types, checks engineered features, and ensures the dataset is ready for exploratory analysis, SQL modelling, Tableau dashboards, and revenue management simulations.

In [36]:
# ==========================================================
# Final Dataset Overview
# ==========================================================

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

df.head()

Rows    : 119,390
Columns : 49


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,stay_duration,departure_date,total_guests,booking_source,lead_time_bucket,arrival_season,weekend_arrival,arrival_month_number,arrival_day_name,guest_nights,weekend_stay,room_nights,booking_status,arrival_quarter,arrival_year_month,family_type
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,0,2015-07-01,2,Direct,90+ Days,Summer,Weekday,7,Wednesday,0,No,0,Completed,Q3,2015-07,Couple
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,0,2015-07-01,2,Direct,90+ Days,Summer,Weekday,7,Wednesday,0,No,0,Completed,Q3,2015-07,Couple
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01,1,2015-07-02,1,Direct,0-7 Days,Summer,Weekday,7,Wednesday,1,No,1,Completed,Q3,2015-07,Solo
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01,1,2015-07-02,1,Travel Agent,8-30 Days,Summer,Weekday,7,Wednesday,1,No,1,Completed,Q3,2015-07,Solo
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03,2015-07-01,2,2015-07-03,2,Travel Agent,8-30 Days,Summer,Weekday,7,Wednesday,4,No,2,Completed,Q3,2015-07,Couple


In [37]:
# ==========================================================
# Missing Values Validation
# ==========================================================

missing_summary = (
    df.isnull()
      .sum()
      .to_frame("Missing Values")
)

missing_summary["Missing %"] = (
    missing_summary["Missing Values"] / len(df) * 100
).round(2)

missing_summary[
    missing_summary["Missing Values"] > 0
]

,Missing Values,Missing %
agent,16340,13.69
company,112593,94.31


### Observations

- Missing values in **children** and **country** have been handled.
- Missing values remaining in **agent** and **company** have been intentionally preserved because they carry business meaning.
- No unexpected missing values were introduced during feature engineering.

In [38]:
# ==========================================================
# Data Types Summary
# ==========================================================

df.dtypes.value_counts()

int64             20
object            10
float64            4
datetime64[ns]     3
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
int32              1
Name: count, dtype: int64

In [39]:
# ==========================================================
# Engineered Features Validation
# ==========================================================

engineered_features = [
    "arrival_date",
    "departure_date",
    "stay_duration",
    "total_guests",
    "booking_source",
    "lead_time_bucket",
    "arrival_season",
    "weekend_arrival",
    "arrival_month_number",
    "arrival_day_name",
    "room_nights",
    "guest_nights",
    "weekend_stay",
    "booking_status",
    "family_type",
    "arrival_quarter",
    "arrival_year_month"
]

validation = pd.DataFrame({
    "Feature": engineered_features,
    "Exists": [col in df.columns for col in engineered_features]
})

validation

,Feature,Exists
0,arrival_date,True
1,departure_date,True
2,stay_duration,True
3,total_guests,True
4,booking_source,True
5,lead_time_bucket,True
6,arrival_season,True
7,weekend_arrival,True
8,arrival_month_number,True
9,arrival_day_name,True


### Validation Summary

The engineered features were successfully created and integrated into the analytical dataset.

These attributes provide standardized business dimensions for customer segmentation, temporal analysis, occupancy modelling, and pricing analytics, eliminating the need for repetitive transformations in downstream notebooks.

In [40]:
# ==========================================================
# Final Dataset Information
# ==========================================================

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 49 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   hotel                           119390 non-null  category      
 1   is_canceled                     119390 non-null  int64         
 2   lead_time                       119390 non-null  int64         
 3   arrival_date_year               119390 non-null  int64         
 4   arrival_date_month              119390 non-null  object        
 5   arrival_date_week_number        119390 non-null  int64         
 6   arrival_date_day_of_month       119390 non-null  int64         
 7   stays_in_weekend_nights         119390 non-null  int64         
 8   stays_in_week_nights            119390 non-null  int64         
 9   adults                          119390 non-null  int64         
 10  children                        119390 non-null  float64

In [41]:
# ==========================================================
# Dataset Summary
# ==========================================================

summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Numerical Columns",
        "Categorical Columns",
        "Date Columns"
    ],
    "Value": [
        df.shape[0],
        df.shape[1],
        len(df.select_dtypes(include="number").columns),
        len(df.select_dtypes(include=["object", "category"]).columns),
        len(df.select_dtypes(include="datetime").columns)
    ]
})

summary

,Metric,Value
0,Rows,119390
1,Columns,49
2,Numerical Columns,25
3,Categorical Columns,21
4,Date Columns,3


## 9. Export Clean Dataset

The cleaned and feature-engineered dataset is exported for use in subsequent stages of the project.

The exported dataset serves as the single source of truth for exploratory analysis, SQL business analytics, Tableau dashboards, and revenue management simulations.

In [43]:
# ==========================================================
# Export Clean Dataset
# ==========================================================

df.to_csv(
    "../data/processed/hotel_bookings_cleaned.csv",
    index=False
)

df.to_parquet(
    "../data/processed/hotel_bookings_cleaned.parquet",
    index=False
)

print("Dataset exported successfully.")

Dataset exported successfully.


# Notebook Summary

### Cleaning Tasks Completed

- ✅ Handled missing values in `children` and `country`
- ✅ Preserved business-relevant missing values in `agent` and `company`
- ✅ Validated duplicate reservations and retained them due to the absence of a unique booking identifier
- ✅ Optimized data types for analytical efficiency

---

### Engineered Features

The following analytical features were created:

| Category | Features |
|----------|----------|
| Date Features | Arrival Date, Departure Date, Stay Duration |
| Customer & Booking | Total Guests, Booking Source, Lead Time Bucket, Arrival Season |
| Operational & Analytical | Weekend Arrival, Arrival Month Number, Arrival Day Name, Room Nights, Guest Nights, Weekend Stay, Booking Status, Family Type, Arrival Quarter, Arrival Year-Month |

---

### Deliverables

- ✅ Cleaned analytical dataset
- ✅ Feature-engineered dataset
- ✅ CSV export
- ✅ Parquet export

---

### Next Steps

The next notebook focuses on **Exploratory Data Analysis (EDA)** to identify booking behaviour, demand patterns, cancellations, occupancy drivers, and pricing opportunities that will support the development of the revenue management decision support system.